# Fault-Injection / Resilience Evidence

This notebook exercises SC-NeuroCore's seeded fault-injection resilience mode for stochastic-computing bitstream layers.

## Evidence Boundary

This notebook uses local bitstream arrays and engineering stress presets only. It does not claim radiation qualification, mission acceptance, hardware SEU measurements, or certified fault tolerance. Real deployment requires target-device fault campaigns, environment-specific radiation analysis, timing/power evidence, and safety-case review.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

from sc_neurocore.fault_injection import (
    DegradationAction,
    FaultInjectionResilienceMode,
    FaultModel,
    RadiationProfile,
    ResilienceModeConfig,
)
from sc_neurocore.fault_injection.resilience_policy import GracefulDegradationPolicy

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

bitstreams = np.array(
    [
        [0, 1, 0, 1, 1, 0, 0, 1],
        [1, 0, 0, 1, 0, 1, 1, 0],
    ],
    dtype=np.uint8,
)
mode = FaultInjectionResilienceMode(
    ResilienceModeConfig(
        layer_id="layer0",
        radiation_profile=RadiationProfile("test", 0.25, "deterministic stress"),
        fault_models=(FaultModel.BIT_FLIP,),
        num_trials=16,
        seed=7,
        policy=GracefulDegradationPolicy(
            warning_affected_ratio=0.01,
            critical_affected_ratio=0.9,
        ),
    )
)
report = mode.run(bitstreams)
trial = report.trial_reports[0]
stress_summary = {
    "layer_id": report.layer_id,
    "input_shape": list(report.input_shape),
    "nominal_probability": report.nominal_probability,
    "recommended_action": report.recommended_action.value,
    "expected_affected_bits": trial.expected_affected_bits,
    "observed_mean_affected_bits": trial.observed_mean_affected_bits,
    "mean_probability_error": trial.mean_probability_error,
    "policy_action": trial.degradation_plan.action.value,
}
assert stress_summary["nominal_probability"] == 0.5
assert stress_summary["recommended_action"] == "extend_bitstream"
stress_summary

In [ ]:
rng = np.random.default_rng(123)
det_bitstreams = rng.integers(0, 2, size=(4, 32), dtype=np.uint8)
det_config = ResilienceModeConfig(
    layer_id="deterministic",
    radiation_profile=RadiationProfile("test", 0.1, "deterministic stress"),
    fault_models=(FaultModel.STUCK_AT_0, FaultModel.DROPOUT),
    num_trials=8,
    seed=99,
)
first = FaultInjectionResilienceMode(det_config).run(det_bitstreams).to_dict()
second = FaultInjectionResilienceMode(det_config).run(det_bitstreams).to_dict()
determinism_summary = {
    "same_seed_bit_exact_report": first == second,
    "fault_models": [item["fault_model"] for item in first["trial_reports"]],
    "trial_count_per_model": [item["num_trials"] for item in first["trial_reports"]],
}
assert determinism_summary["same_seed_bit_exact_report"] is True
determinism_summary

In [ ]:
correlated = np.tile(np.array([1, 0, 1, 0, 1, 0, 1, 0], dtype=np.uint8), (4, 1))
replay_report = FaultInjectionResilienceMode(
    ResilienceModeConfig(
        layer_id="correlated",
        radiation_profile=RadiationProfile("zero", 0.0, "no injected faults"),
        fault_models=(FaultModel.BIT_FLIP,),
        num_trials=4,
        seed=11,
    )
).run(correlated)
replay_summary = {
    "recommended_action": replay_report.recommended_action.value,
    "requires_replay": replay_report.requires_replay,
    "replay_seed": replay_report.trial_reports[0].degradation_plan.replay_seed,
}
assert replay_report.recommended_action == DegradationAction.REPLAY_WITH_SEED
assert replay_summary["replay_seed"] == 11
replay_summary

In [ ]:
profiles = [
    RadiationProfile.terrestrial(),
    RadiationProfile.leo(),
    RadiationProfile.geo(),
    RadiationProfile.deep_space(),
]
radiation_profile_summary = [
    {"name": profile.name, "ber": profile.ber, "description": profile.description}
    for profile in profiles
]
assert [item["ber"] for item in radiation_profile_summary] == sorted(
    item["ber"] for item in radiation_profile_summary
)
radiation_profile_summary

In [ ]:
bad_mode = FaultInjectionResilienceMode(
    ResilienceModeConfig(layer_id="bad", radiation_profile=RadiationProfile("test", 0.0))
)
try:
    bad_mode.run(np.array([[0, 2]], dtype=np.uint8))
except ValueError as exc:
    invalid_bits_refusal = str(exc)
else:
    raise AssertionError("non-binary bitstreams were accepted")

try:
    FaultInjectionResilienceMode(
        ResilienceModeConfig(layer_id="bad-shape", radiation_profile=RadiationProfile("test", 0.0))
    ).run(np.zeros(8, dtype=np.uint8))
except ValueError as exc:
    bad_shape_refusal = str(exc)
else:
    raise AssertionError("one-dimensional bitstream was accepted")

guardrail_summary = {
    "invalid_bits_refusal": invalid_bits_refusal,
    "bad_shape_refusal": bad_shape_refusal,
}
assert "0/1" in invalid_bits_refusal
assert "shape" in bad_shape_refusal
guardrail_summary

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.fault-resilience-evidence.v1",
    "stress_summary": stress_summary,
    "determinism_summary": determinism_summary,
    "replay_summary": replay_summary,
    "radiation_profile_summary": radiation_profile_summary,
    "guardrails": guardrail_summary,
    "evidence_boundary": "Local seeded fault-injection evidence only; no radiation qualification, mission acceptance, hardware SEU, or certified fault-tolerance claim.",
}
manifest